# recs_XXX_eval_two_stage_habit_session

Purpose: evaluate retrieval approaches under one consistent contract.

Evaluation principle: ranking quality and personalization quality are co-primary and are evaluated together for every method comparison.

Method families (explicit):
- Family 1 (incumbent single-stage): `raw`, `popularity_train`, `multi_mean_train`
- Family 2 (single-stage fused): `fused_single_stage = score(normalize(alpha*u_behavior + beta*u_reviews + gamma*q_session), item)`
- Family 3 (retrieval→ranking cascade): `habit_session_two_stage = rerank_topM(score(u_habit, item), q_session)`

Similarity/score note:
- The model computes a similarity value between query/user vector and item vector; that similarity is the item score.
- Items are sorted in descending score order for ranking metrics.

Vector definitions:
- `q_session`: embedding of the current query review text.
- `u_reviews`: pooled embedding of support/train review texts for the user.
- `u_behavior`: pooled embedding of support/train app IDs via the item embedding matrix.
- `u_habit`: retrieval-stage habit vector used for candidate generation (in this notebook: `u_habit = habit_fused = normalize(0.5*u_behavior + 0.5*u_reviews)`).

Pipeline definitions:
- Retrieval stage: candidate generation using habit vector (`u_habit`) against item embeddings; evaluate candidate hit/recall at `M_STAGE1`.
- Ranking stage: rerank retrieval-stage candidates with session/query signal (`q_session` or fused variant); evaluate final ranking metrics at `K_FINAL`.

Decile diagnostics (for both retrieval-stage and ranking-stage views):
- Popularity deciles are computed from target-item train popularity (`app_pos_count_train`) and binned with `qcut` into `1..10`.
- `1` = least popular (long tail), `10` = most popular (head).
- Report per-decile metrics and deltas vs anchor methods to expose head-vs-tail behavior.

Execution order:
1. Setup and config
2. Shared helpers
3. Build eval examples and metadata
4. Build vectors (`u_behavior`, `u_reviews`, `q_session`, fused)
5. Retrieval-stage evaluation
6. Ranking-stage evaluation + incumbent comparison
7. Support/pop-decile slices + baseline deltas
8. Personalization metrics integrated in overall/slice/support/pop-decile tables
9. Write summary artifact

This notebook is intentionally lean and artifact-driven.
Reference docs:
- `docs/eval_contract.md`
- `docs/recommender_transition_plan.md`

## 1) Setup and Top-Level Config

In [12]:
from pathlib import Path

import numpy as np
import pandas as pd

from steam_review_ml.recommender import evaluation as ev


def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for d in [here, *here.parents]:
        if (d / "pyproject.toml").is_file():
            return d
    raise RuntimeError(f"Could not find repo root from start={start}")


# ===== 1) Setup and top-level config =====
REPO_ROOT = _find_repo_root(Path.cwd())
ARTIFACT_DIR = REPO_ROOT / "artifacts" / "recs"
EVAL_DIR = ARTIFACT_DIR / "eval"

SPLIT = "val"
K_FINAL = 10
M_STAGE1 = 100
K_PERSONALIZATION = 10
MAX_EXAMPLES = 12_500
RANDOM_SEED = 2026

# Keep notebook sampling aligned with configs/recs_job_eval_retrieval.json
COHORT_SIZING = {
    ("val_multi_pos_eval", "val_multi_pos_train"): 0.5,
    ("val_multi_pos_eval", "val_pos_train"): 0.25,
    ("val_multi_pos_eval", "val_train"): 0.15,
    ("val_multi_pos_eval", "val_no_train"): 0.1,
}

ALPHA_BEHAVIOR = 0.45
BETA_REVIEWS = 0.45
GAMMA_SESSION = 0.10

METHODS_INCUMBENT = ["raw", "popularity_train", "multi_mean_train"]
SUPPORT_BUCKETS = ["0", "1", "2-3", "4-7", "8+"]
SLICE_RULES = {
    "slice_a_multi_target": "n_eval_targets >= 2",
    "slice_b_single_target": "n_eval_targets == 1",
    "slice_c_zero_target": "n_eval_targets == 0",
}

ENABLE_PLOTS = False

## 2) Shared Helpers

In [13]:
# ===== 2) Shared helpers (single definition source) =====

def l2_normalize(v: np.ndarray) -> np.ndarray:
    arr = np.asarray(v, dtype=np.float32).ravel()
    nrm = float(np.linalg.norm(arr))
    if nrm <= 1e-12:
        return arr
    return (arr / nrm).astype(np.float32)


def support_bucket(n: int) -> str:
    n = int(n)
    if n <= 0:
        return "0"
    if n == 1:
        return "1"
    if n <= 3:
        return "2-3"
    if n <= 7:
        return "4-7"
    return "8+"


def jaccard(a: set[int], b: set[int]) -> float:
    u = a | b
    if not u:
        return 1.0
    return len(a & b) / len(u)


def rank_rows(scores: np.ndarray) -> np.ndarray:
    return np.argsort(-scores)


def scores_from_query(q: np.ndarray, X: np.ndarray, app_to_row: dict[int, int], query_app_id: int) -> np.ndarray:
    s = (X @ q).astype(np.float32)
    row = app_to_row.get(int(query_app_id))
    if row is not None:
        s[row] = -np.inf
    return s


def eval_row(ranked: np.ndarray, positives: set[int], app_ids: np.ndarray, k: int) -> dict[str, float]:
    return {
        "Hit@K": ev.hit_rate_at_k(ranked, positives, k, app_ids),
        "Recall@K": ev.recall_at_k(ranked, positives, k, app_ids),
        "MAP@K": ev.average_precision_at_k(ranked, positives, k, app_ids),
        "NDCG@K": ev.ndcg_at_k(ranked, positives, k, app_ids),
        "MRR": ev.mrr(ranked, positives, app_ids),
    }


def slice_name_from_n_targets(n_eval_targets: int) -> str:
    if n_eval_targets >= 2:
        return "slice_a_multi_target"
    if n_eval_targets == 1:
        return "slice_b_single_target"
    return "slice_c_zero_target"

## 3) Build Evaluation Examples and Metadata

In [14]:
# ===== 3) Build evaluation examples and metadata =====
inputs = ev.prepare_eval_inputs(
    repo_root=REPO_ROOT,
    split=SPLIT,
    active_cohort="all",
    max_examples=MAX_EXAMPLES,
    support_app_filter_mode="strict",
    cohort_sizing=COHORT_SIZING,
    min_review_chars=30,
    max_train_rows_per_user=5,
    random_seed=RANDOM_SEED,
    artifact_dir=ARTIFACT_DIR,
    verbose=True,
)

examples = inputs.examples
X = inputs.embedding_matrix
app_ids = inputs.app_ids
app_to_row = inputs.app_to_row
retriever = inputs.retriever

example_meta = pd.DataFrame(
    {
        "ex_idx": np.arange(len(examples), dtype=int),
        "user_id": [str(ex["user_id"]) for ex in examples],
        "query_app_id": [int(ex["query_app_id"]) for ex in examples],
        "n_eval_targets": [int(ex["n_eval_targets"]) for ex in examples],
        "n_support_train": [int(len(ex.get("support_texts_train", []))) for ex in examples],
    }
)
example_meta["slice_name"] = example_meta["n_eval_targets"].map(slice_name_from_n_targets)
example_meta["train_support_bucket"] = example_meta["n_support_train"].map(support_bucket)

print("examples:", len(examples))
display(example_meta.head())

Loaded split rows: eval=1,617,344 train=3,878,131 split_used=val


build eval examples: 100%|██████████| 12500/12500 [00:00<00:00, 26352.57row/s]


Prepared evaluation inputs: records=1,400,227 sampled=12,500 evaluable_examples=12,500
Drop reasons: {'no_other_positive_app': 0}
examples: 12500


,ex_idx,user_id,query_app_id,n_eval_targets,n_support_train,slice_name,train_support_bucket
0,0,76561198001296435,812140,1,2,slice_b_single_target,2-3
1,1,76561198006360052,485510,1,5,slice_b_single_target,4-7
2,2,76561198094909231,646570,2,5,slice_a_multi_target,4-7
3,3,76561198133278614,4000,1,2,slice_b_single_target,2-3
4,4,76561198073849336,582010,1,3,slice_b_single_target,2-3


In [15]:
pd.DataFrame(examples).head()

,user_id,query_app_id,query_text,query_ts,positives,n_eval_targets,support_texts_train,train_review_rows,cohort,eval_pos_cohort
0,76561198001296435,812140,"> Got it on Sale\n> Started it up, played Kass...",1.568428e+09,{262060},1,[Facepalm as your teammates run into marked en...,"[{'app_id': 359550, 'text': 'Facepalm as your ...",val_multi_pos_train,val_multi_pos_eval
1,76561198006360052,485510,Pretty fun so far. Took me about 5 hours to b...,1.602431e+09,{552500},1,"[As a long term Monster Hunter fan, this itche...","[{'app_id': 899440, 'text': 'As a long term Mo...",val_multi_pos_train,val_multi_pos_eval
2,76561198094909231,646570,A game about setting up your broken-a.-f. card...,1.602966e+09,"{40800, 435150}",2,[A modern classic for single-player narrative ...,"[{'app_id': 8870, 'text': 'A modern classic fo...",val_multi_pos_train,val_multi_pos_eval
3,76561198133278614,4000,You can do almost anything in this game 10/10,1.528897e+09,{413150},1,[Man i cna't tell you how good this game is yo...,"[{'app_id': 253230, 'text': 'Man i cna't tell ...",val_multi_pos_train,val_multi_pos_eval
4,76561198073849336,582010,My first Monster Hunter was MH3: Tri. I was wa...,1.535904e+09,{814380},1,"[*Music starts* Unicorns, rainbows, dinosaurs...","[{'app_id': 221640, 'text': '*Music starts* U...",val_multi_pos_train,val_multi_pos_eval


In [16]:
example_meta['slice_name'].value_counts()

slice_name
slice_b_single_target    11775
slice_a_multi_target       725
Name: count, dtype: int64

## 4) Build Vectors (`u_behavior`, `u_reviews`, `q_session`, `habit_fused`)

In [17]:
# ===== 4) Build vectors (u_behavior, u_reviews, q_session, fused) =====
vector_rows = []
for ex_idx, ex in enumerate(examples):
    q_session = retriever.embed_text(str(ex["query_text"]))

    support_texts = [str(t).strip() for t in ex.get("support_texts_train", []) if str(t).strip()]
    if support_texts:
        support_vecs = np.stack([retriever.embed_text(t) for t in support_texts], axis=0).astype(np.float32)
        u_reviews = l2_normalize(support_vecs.mean(axis=0))
    else:
        u_reviews = q_session

    support_rows = ex.get("train_review_rows", [])
    support_app_ids = sorted({int(r["app_id"]) for r in support_rows if int(r["app_id"]) in app_to_row})
    if support_app_ids:
        emb = np.stack([X[app_to_row[a]] for a in support_app_ids], axis=0).astype(np.float32)
        u_behavior = l2_normalize(emb.mean(axis=0))
    else:
        u_behavior = u_reviews

    q_fused_user_session = l2_normalize(
        ALPHA_BEHAVIOR * u_behavior + BETA_REVIEWS * u_reviews + GAMMA_SESSION * q_session
    )
    habit_fused = l2_normalize(0.5 * u_behavior + 0.5 * u_reviews)

    vector_rows.append(
        {
            "ex_idx": ex_idx,
            "q_session": q_session,
            "u_reviews": u_reviews,
            "u_behavior": u_behavior,
            "habit_fused": habit_fused,
            "q_fused_user_session": q_fused_user_session,
        }
    )

vector_store = {int(r["ex_idx"]): r for r in vector_rows}
print("vectorized examples:", len(vector_store))

vectorized examples: 12500


## 5) Retrieval-Stage Evaluation (Candidate Generation at M)

In [18]:
# ===== 5) Retrieval-stage evaluation (candidate generation at M) =====
stage1_method_vectors = {
    "stage1_u_behavior": "u_behavior",
    "stage1_u_reviews": "u_reviews",
    "stage1_habit_fused": "habit_fused",
}

stage1_rows = []
for ex_idx, ex in enumerate(examples):
    positives = set(int(a) for a in ex["positives"])
    if not positives:
        continue
    for method_name, vec_key in stage1_method_vectors.items():
        q = vector_store[ex_idx][vec_key]
        ranked = rank_rows(scores_from_query(q, X, app_to_row, int(ex["query_app_id"])))
        topM = ranked[:M_STAGE1]
        rec_m = ev.recall_at_k(topM, positives, M_STAGE1, app_ids)
        hit_m = ev.hit_rate_at_k(topM, positives, M_STAGE1, app_ids)
        stage1_rows.append(
            {
                "method": method_name,
                "ex_idx": ex_idx,
                "Recall@M": rec_m,
                "Hit@M": hit_m,
            }
        )

stage1_per_example = pd.DataFrame(stage1_rows)
stage1_table = (
    stage1_per_example.groupby("method", observed=True)[["Recall@M", "Hit@M"]]
    .mean()
    .reset_index()
    .sort_values(["Recall@M", "Hit@M"], ascending=False)
)

display(stage1_table)

,method,Recall@M,Hit@M
1,stage1_u_behavior,0.484793,0.50280
0,stage1_habit_fused,0.484683,0.50256
2,stage1_u_reviews,0.452953,0.47120


## 6) Ranking-Stage Evaluation (Rerank) + Incumbent Baselines

In [19]:
# ===== 6) Ranking-stage evaluation (rerank) + incumbent baselines =====
rng = np.random.default_rng(RANDOM_SEED)
inc_registry = ev._build_method_registry(
    retriever=retriever,
    X=X,
    pop_row=inputs.pop_row,
    app_to_row=app_to_row,
    multi_max_reviews=5,
    rng=rng,
    mask_query_app=True,
)
inc_registry = {m: inc_registry[m] for m in METHODS_INCUMBENT}


def _single_stage_scores(ex_idx: int, ex: dict) -> dict[str, np.ndarray]:
    """Return full-catalog scores for incumbent + fused single-stage methods."""
    return {
        **{m: inc_registry[m](ex) for m in METHODS_INCUMBENT},
        "single_fused_user_session": scores_from_query(
            vector_store[ex_idx]["q_fused_user_session"], X, app_to_row, int(ex["query_app_id"])
        ),
    }


def _two_stage_rerank_order(ex_idx: int, ex: dict, q_stage1: np.ndarray) -> np.ndarray:
    """Retrieve top-M from stage-1 vector and rerank those candidates by session."""
    s1 = scores_from_query(q_stage1, X, app_to_row, int(ex["query_app_id"]))
    cands = rank_rows(s1)[:M_STAGE1]
    q_session = vector_store[ex_idx]["q_session"]
    s2 = scores_from_query(q_session, X, app_to_row, int(ex["query_app_id"]))
    return cands[np.argsort(-s2[cands])]


def _score_fn_for_method(method_name: str):
    """Build a scorer callable that mirrors each method's inference behavior."""
    if method_name in METHODS_INCUMBENT:
        return lambda ex, m=method_name: inc_registry[m](ex)

    if method_name == "single_fused_user_session":
        return lambda ex, _m=method_name: scores_from_query(
            vector_store[int(ex["_ex_idx"])]["q_fused_user_session"], X, app_to_row, int(ex["query_app_id"])
        )

    mapping = {
        "two_stage_behavior_session": "u_behavior",
        "two_stage_reviews_session": "u_reviews",
        "two_stage_habit_fused_session": "habit_fused",
    }
    if method_name in mapping:
        key = mapping[method_name]

        def _fn(ex, _k=key):
            ex_idx = int(ex["_ex_idx"])
            q_session = vector_store[ex_idx]["q_session"]
            q_stage1 = vector_store[ex_idx][_k]
            s1 = scores_from_query(q_stage1, X, app_to_row, int(ex["query_app_id"]))
            cands = rank_rows(s1)[:M_STAGE1]
            s2 = scores_from_query(q_session, X, app_to_row, int(ex["query_app_id"]))
            out = np.full_like(s2, -np.inf)
            out[cands] = s2[cands]
            return out

        return _fn

    raise KeyError(method_name)


def _examples_for_personalization(examples: list[dict]) -> list[dict]:
    """Attach example indices so scorer callables can access vector_store."""
    out = []
    for ex_idx, ex in enumerate(examples):
        ex_copy = dict(ex)
        ex_copy["_ex_idx"] = ex_idx
        out.append(ex_copy)
    return out


rows = []
for ex_idx, ex in enumerate(examples):
    positives = set(int(a) for a in ex["positives"])
    if not positives:
        continue

    for method_name, s in _single_stage_scores(ex_idx, ex).items():
        ranked = rank_rows(s)
        metric_vals = eval_row(ranked, positives, app_ids, K_FINAL)
        rows.append(
            {
                "method": method_name,
                "family": "incumbent" if method_name in METHODS_INCUMBENT else "single_fused",
                "ex_idx": ex_idx,
                **metric_vals,
            }
        )

    two_stage_sources = {
        "two_stage_behavior_session": vector_store[ex_idx]["u_behavior"],
        "two_stage_reviews_session": vector_store[ex_idx]["u_reviews"],
        "two_stage_habit_fused_session": vector_store[ex_idx]["habit_fused"],
    }
    for method_name, q_stage1 in two_stage_sources.items():
        rerank_order = _two_stage_rerank_order(ex_idx, ex, q_stage1)
        metric_vals = eval_row(rerank_order, positives, app_ids, K_FINAL)
        rows.append(
            {
                "method": method_name,
                "family": "two_stage",
                "ex_idx": ex_idx,
                **metric_vals,
            }
        )

per_example_table = pd.DataFrame(rows).merge(example_meta, on="ex_idx", how="left")

overall_table = (
    per_example_table.groupby(["family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
    .sort_values(["NDCG@K", "MAP@K", "MRR"], ascending=False)
)

by_slice_table = (
    per_example_table.groupby(["slice_name", "family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
)

all_methods = sorted(per_example_table["method"].unique().tolist())
examples_for_personalization = _examples_for_personalization(examples)
methods_for_personalization = {m: _score_fn_for_method(m) for m in all_methods}

personalization_table = ev._table_personalization(
    methods=methods_for_personalization,
    examples=examples_for_personalization,
    X=X,
    app_ids=app_ids,
    pop_row=inputs.pop_row,
    k_personalization=K_PERSONALIZATION,
    verbose=False,
)

slice_personalization_rows = []
for slice_name, g in per_example_table.groupby("slice_name", observed=True):
    ex_indices = sorted({int(x) for x in g["ex_idx"].tolist()})
    p = ev._table_personalization(
        methods=methods_for_personalization,
        examples=examples_for_personalization,
        X=X,
        app_ids=app_ids,
        pop_row=inputs.pop_row,
        k_personalization=K_PERSONALIZATION,
        example_indices=ex_indices,
        verbose=False,
    ).copy()
    p["slice_name"] = str(slice_name)
    slice_personalization_rows.append(p)

slice_personalization = pd.concat(slice_personalization_rows, ignore_index=True, sort=False)

overall_table = overall_table.merge(personalization_table, on="method", how="left")
by_slice_table = by_slice_table.merge(slice_personalization, on=["method", "slice_name"], how="left")

display(overall_table.head(20))

KeyboardInterrupt: 

## 7) Cross-Sections: Support Buckets, Pop Deciles, and Deltas

In [ ]:
# ===== 7) Cross-sections: support buckets, pop deciles, and deltas =====

def _build_ex_pop_table(examples: list[dict], app_ids: np.ndarray, pop_row: np.ndarray) -> pd.DataFrame:
    """Compute mean target popularity per example and assign popularity deciles."""
    app_pop = {int(a): float(c) for a, c in zip(app_ids, pop_row)}
    rows = []
    for ex_idx, ex in enumerate(examples):
        vals = [app_pop.get(int(a), 0.0) for a in ex["positives"]]
        rows.append({"ex_idx": ex_idx, "pos_pop_mean": float(np.mean(vals)) if vals else np.nan})
    ex_pop = pd.DataFrame(rows)
    valid = ex_pop["pos_pop_mean"].notna()
    if valid.sum() > 0:
        ex_pop.loc[valid, "pos_pop_decile"] = pd.qcut(
            ex_pop.loc[valid, "pos_pop_mean"], q=10, labels=[f"D{i}" for i in range(1, 11)], duplicates="drop"
        )
    return ex_pop


def _personalization_by_group(
    per_example_table: pd.DataFrame,
    group_col: str,
    methods_for_personalization: dict,
    examples_for_personalization: list[dict],
) -> pd.DataFrame:
    """Compute personalization table at the same grouping level as ranking metrics."""
    rows = []
    for group_val, g in per_example_table.groupby(group_col, observed=True):
        ex_indices = sorted({int(x) for x in g["ex_idx"].tolist()})
        p = ev._table_personalization(
            methods=methods_for_personalization,
            examples=examples_for_personalization,
            X=X,
            app_ids=app_ids,
            pop_row=inputs.pop_row,
            k_personalization=K_PERSONALIZATION,
            example_indices=ex_indices,
            verbose=False,
        ).copy()
        p[group_col] = group_val
        rows.append(p)
    return pd.concat(rows, ignore_index=True, sort=False)


def _build_delta_vs_baselines(overall_table: pd.DataFrame) -> pd.DataFrame:
    """Compute method deltas vs raw and popularity anchors."""
    anchors = overall_table[overall_table["method"].isin(["raw", "popularity_train"])][["method", "Hit@K", "NDCG@K", "MRR"]]
    anchor_map = {r["method"]: r for _, r in anchors.iterrows()}
    rows = []
    for _, r in overall_table.iterrows():
        for anchor in ["raw", "popularity_train"]:
            if anchor not in anchor_map:
                continue
            a = anchor_map[anchor]
            rows.append(
                {
                    "method": r["method"],
                    "family": r["family"],
                    "anchor": anchor,
                    "Hit@10_delta_vs_anchor": float(r["Hit@K"] - a["Hit@K"]),
                    "NDCG@10_delta_vs_anchor": float(r["NDCG@K"] - a["NDCG@K"]),
                    "MRR_delta_vs_anchor": float(r["MRR"] - a["MRR"]),
                }
            )
    return pd.DataFrame(rows)


by_support_table = (
    per_example_table.groupby(["train_support_bucket", "family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
)

ex_pop = _build_ex_pop_table(examples, app_ids, inputs.pop_row)

by_pop_decile_table = (
    per_example_table.merge(ex_pop[["ex_idx", "pos_pop_decile"]], on="ex_idx", how="left")
    .dropna(subset=["pos_pop_decile"])
    .groupby(["pos_pop_decile", "family", "method"], observed=True)[["Hit@K", "Recall@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
)

support_personalization = _personalization_by_group(
    per_example_table=per_example_table,
    group_col="train_support_bucket",
    methods_for_personalization=methods_for_personalization,
    examples_for_personalization=examples_for_personalization,
)

per_ex_pop = per_example_table.merge(ex_pop[["ex_idx", "pos_pop_decile"]], on="ex_idx", how="left")
pop_personalization = _personalization_by_group(
    per_example_table=per_ex_pop.dropna(subset=["pos_pop_decile"]),
    group_col="pos_pop_decile",
    methods_for_personalization=methods_for_personalization,
    examples_for_personalization=examples_for_personalization,
)

by_support_table = by_support_table.merge(support_personalization, on=["method", "train_support_bucket"], how="left")
by_pop_decile_table = by_pop_decile_table.merge(pop_personalization, on=["method", "pos_pop_decile"], how="left")

delta_vs_baselines_table = _build_delta_vs_baselines(overall_table)

display(by_support_table.head(20))
display(by_pop_decile_table.head(20))
display(delta_vs_baselines_table.head(20))

## 8) (Integrated Above) Personalization Metrics (Co-Primary)

In [ ]:
# Personalization metrics are integrated into Sections 6 and 7 tables (overall/slice/support/pop-decile).
# This cell is intentionally empty.

## 9) Write Summary Artifact + Compact Family Summary

In [ ]:
# ===== 9) Write summary artifact + compact family summary =====
OUTPUT_PATH = ARTIFACT_DIR / "eval_two_stage_summary.csv"

summary_tables = {
    "stage1": stage1_table,
    "overall": overall_table,
    "by_slice": by_slice_table,
    "by_support": by_support_table,
    "by_pop_decile": by_pop_decile_table,
    "delta_vs_baselines": delta_vs_baselines_table,
}

for name, table in summary_tables.items():
    print(f"{name:>18}: rows={len(table)}")

combined = []
for name, table in summary_tables.items():
    t = table.copy()
    t.insert(0, "section", name)
    combined.append(t)

summary_artifact = pd.concat(combined, ignore_index=True, sort=False)
summary_artifact.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote summary artifact: {OUTPUT_PATH}")

# Optional compact family-level summary
family_best = (
    overall_table.sort_values(["family", "NDCG@K", "MAP@K", "MRR"], ascending=[True, False, False, False])
    .groupby("family", as_index=False)
    .head(1)
)
display(family_best)